In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("PortDataCleaning").master("local[*]").getOrCreate()
df = spark.read.option("header", True).option("inferSchema", True).parquet("../data/processed/port_activity_clean.parquet/part-00000-bb1287fb-8f0f-43ec-9c16-f683e9bef454-c000.snappy.parquet")
df.show()

+-------------------+----+-----+---+--------+-----------+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+
|               date|year|month|day|  portid|   portname|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|
+-------------------+----+-----+---+--------+-----------+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+----------

In [14]:
annual_analysis = df.groupBy("year", "portname").agg(
    sum("portcalls").alias("total_port_calls"),
    sum("import").alias("total_imports"),
    sum("export").alias("total_exports")
).orderBy("year", "portname")
annual_analysis.show()
#annual_analysis.write.mode("overwrite").csv("../data/processed/annual_analysis")

+----+-----------+----------------+-------------+-------------+
|year|   portname|total_port_calls|total_imports|total_exports|
+----+-----------+----------------+-------------+-------------+
|2019|    Colombo|            4190|     20693145|     10399800|
|2019|      Galle|             142|       122148|        18167|
|2019| Hambantota|             273|       729987|       300964|
|2019|Trincomalee|             177|      3301280|       446358|
|2020|    Colombo|            3566|     18037558|      9405608|
|2020|      Galle|             115|        90415|        10421|
|2020| Hambantota|             295|      1435721|       413294|
|2020|Trincomalee|             147|      2445502|       456631|
|2021|    Colombo|            3596|     18558953|      9936167|
|2021|      Galle|              15|         4125|          627|
|2021| Hambantota|             264|      1194119|       244898|
|2021|Trincomalee|             146|      2711432|       130925|
|2022|    Colombo|            3745|     

In [ ]:
import plotly.express as px

plot_df = annual_analysis.toPandas()

fig = px.bar(
    plot_df,
    x="year",2
    y="total_port_calls",
    color="portname",
    barmode="group",
    title="Port Calls by Year and Port",
)
fig.show()

d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [12]:
total_import = px.bar(
    plot_df,
    x="year",
    y="total_imports",
    color="portname",
    barmode="group",
    title="Total Imports"
)
total_import.show()

In [13]:
total_exports = px.bar(
    plot_df,
    x="year",
    y="total_exports",
    color="portname",
    barmode="group",
    title="Total Exports"
)
total_exports.show()

In [14]:
plot_df = annual_analysis.select(
    "year", "portname", "total_imports", "total_exports"
).toPandas()

plot_df = plot_df.sort_values(["year", "portname"])

long_df = plot_df.melt(
    id_vars=["year", "portname"],
    value_vars=["total_imports", "total_exports"],
    var_name="trade_type",
    value_name="volume",
)

fig = px.bar(
    long_df,
    x="year",
    y="volume",
    color="trade_type",
    barmode="group",
    facet_col="portname",
    facet_col_wrap=2,
    title="Total Imports and Exports by Year and Port",
    labels={
        "year": "Year",
        "volume": "Cargo Volume",
        "trade_type": "Trade Type",
        "portname": "Port",
    },
)

fig.show()

d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
import_analysis_by_type = df.groupBy("year", "portname").agg(
    sum("import_container").alias("total_import_container"),
    sum("import_dry_bulk").alias("totl_import_dry_bulk"),
    sum("import_general_cargo").alias("total_import_general_cargo"),
    sum("import_roro").alias("total_import_roro"),
    sum("import_tanker").alias("total_import_tanker"),
    sum("import_cargo").alias("total_import_cargo")
).orderBy("year", "portname")

import_analysis_by_type.show()
import_analysis_by_type.count()

+----+-----------+----------------------+--------------------+--------------------------+-----------------+-------------------+------------------+
|year|   portname|total_import_container|totl_import_dry_bulk|total_import_general_cargo|total_import_roro|total_import_tanker|total_import_cargo|
+----+-----------+----------------------+--------------------+--------------------------+-----------------+-------------------+------------------+
|2019|    Colombo|              15122257|             3309374|                    286602|            10005|            1964693|          18728391|
|2019|      Galle|                     0|                   0|                    122148|                0|                  0|            122148|
|2019| Hambantota|                  6604|              572181|                      6282|            88391|              56523|            673464|
|2019|Trincomalee|                     0|             3143881|                     15948|            17418|           

32

In [5]:
import_pd = import_analysis_by_type.toPandas()

value_columns = [
    "total_import_container",
    "total_import_dry_bulk",
    "total_import_general_cargo",
    "total_import_roro",
    "total_import_tanker",
    "total_import_cargo",
]

plot_df = import_pd.rename(
    columns={"totl_import_dry_bulk": "total_import_dry_bulk"}
).melt(
    id_vars=["year", "portname"],
    value_vars=value_columns,
    var_name="cargo_type",
    value_name="imports",
)

fig = px.bar(
    plot_df,
    x="year",
    y="imports",
    color="cargo_type",
    barmode="group",
    facet_col="portname",
    facet_col_wrap=2,
    title="Imports by Cargo Type, Year, and Port",
    labels={"year": "Year", "imports": "Import Volume", "cargo_type": "Cargo Type"},
)

fig.show()

d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [6]:
from pyspark.sql.functions import col, sum as spark_sum

import_columns = [
    "import_container",
    "import_dry_bulk",
    "import_general_cargo",
    "import_roro",
    "import_tanker",
    "import_cargo"
]

colombo_port_data = (
    df.filter(col("portname") == "Colombo")
      .groupBy("year", "month")
      .agg(*[
          spark_sum(col(c)).alias(f"total_{c}")
          for c in import_columns
      ])
      .orderBy("year", "month")
)

colombo_port_data.show(truncate=False)
colombo_port_data.count()

+----+-----+----------------------+---------------------+--------------------------+-----------------+-------------------+------------------+
|year|month|total_import_container|total_import_dry_bulk|total_import_general_cargo|total_import_roro|total_import_tanker|total_import_cargo|
+----+-----+----------------------+---------------------+--------------------------+-----------------+-------------------+------------------+
|2019|1    |1227190               |288942               |25391                     |0                |73437              |1541543           |
|2019|2    |1212284               |232184               |12438                     |900              |106056             |1457819           |
|2019|3    |1354229               |367811               |24105                     |0                |265312             |1746156           |
|2019|4    |1293059               |265844               |12292                     |0                |210927             |1571211           |
|2019|

94

In [7]:
import pandas as pd
import plotly.express as px

plot_df = colombo_port_data.toPandas()

plot_df["date"] = pd.to_datetime(
    plot_df[["year", "month"]].assign(day=1)
)
plot_df = plot_df.sort_values("date")

columns = [
    "total_import_container",
    "total_import_dry_bulk",
    "total_import_general_cargo",
    "total_import_roro",
    "total_import_tanker",
]

fig = px.line(
    plot_df,
    x="date",
    y=columns,
    markers=True,
    title="Monthly Imports at Colombo Port",
    labels={
        "date": "Month",
        "value": "Import volume",
        "variable": "Import category",
    },
)

fig.show()

d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [8]:
from pyspark.sql.functions import col, sum as spark_sum

import_columns = [
    "export_container",
    "export_dry_bulk",
    "export_general_cargo",
    "export_roro",
    "export_tanker",
    "export_cargo"
]

colombo_port_export_data = (
    df.filter(col("portname") == "Colombo")
      .groupBy("year", "month")
      .agg(*[
          spark_sum(col(c)).alias(f"total_{c}")
          for c in import_columns
      ])
      .orderBy("year", "month")
)

colombo_port_export_data.show(truncate=False)
colombo_port_export_data.count()

+----+-----+----------------------+---------------------+--------------------------+-----------------+-------------------+------------------+
|year|month|total_export_container|total_export_dry_bulk|total_export_general_cargo|total_export_roro|total_export_tanker|total_export_cargo|
+----+-----+----------------------+---------------------+--------------------------+-----------------+-------------------+------------------+
|2019|1    |868983                |3360                 |31149                     |0                |9289               |903498            |
|2019|2    |674447                |12276                |25302                     |0                |7904               |712028            |
|2019|3    |837890                |4850                 |25110                     |0                |11925              |867854            |
|2019|4    |805889                |6720                 |34189                     |0                |9022               |846800            |
|2019|

94

In [12]:
plot_export_df = colombo_port_export_data.toPandas()

plot_export_df["date"] = pd.to_datetime(
    plot_df[["year", "month"]].assign(day=1)
)
plot_export_df = plot_df.sort_values("date")

columns = [
    "total_export_container",
    "total_export_dry_bulk",
    "total_export_general_cargo",
    "total_export_roro",
    "total_export_tanker",
]

plot_export_df = colombo_port_export_data.toPandas()

plot_export_df["date"] = pd.to_datetime(
    plot_export_df[["year", "month"]].assign(day=1)
)
plot_export_df = plot_export_df.sort_values("date")

columns = [
    "total_export_container",
    "total_export_dry_bulk",
    "total_export_general_cargo",
    "total_export_roro",
    "total_export_tanker",
]

fig = px.line(
    plot_export_df,
    x="date",
    y=columns,
    markers=True,
    title="Monthly Exports at Colombo Port",
    labels={
        "date": "Month",
        "value": "Export volume",
        "variable": "Export category",
    },
)

fig.show()

d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [21]:
import os

# Create a folder for dashboard data
dashboard_path = "../data/processed/dashboard"

os.makedirs(dashboard_path, exist_ok=True)

# 1. Annual port analysis
annual_analysis.toPandas().to_csv(
    f"{dashboard_path}/annual_analysis.csv",
    index=False
)

# 2. Import analysis by cargo type
import_pd = import_analysis_by_type.toPandas()

# Correct the typo in the existing column name
import_pd = import_pd.rename(
    columns={
        "totl_import_dry_bulk": "total_import_dry_bulk"
    }
)

import_pd.to_csv(
    f"{dashboard_path}/import_analysis_by_type.csv",
    index=False
)

# 3. Colombo monthly import analysis
colombo_port_data.toPandas().to_csv(
    f"{dashboard_path}/colombo_monthly_imports.csv",
    index=False
)

print("Dashboard datasets exported successfully!")

d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Big data project\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Dashboard datasets exported successfully!
